# Create noisy motl lists


In [8]:
import numpy as np
import pandas as pd
from cryocat import cryomotl

In [9]:
def make_noise(N, magnitude):
    """
    Generate N random 3D vectors of fixed length = magnitude,
    uniformly distributed in direction.
    """
    # sample from normal, shape (N,3)
    vecs = np.random.normal(size=(N, 3))
    # normalize each to unit length
    norms = np.linalg.norm(vecs, axis=1, keepdims=True)
    unit_vecs = vecs / norms
    # scale to desired magnitude
    return unit_vecs * magnitude



In [21]:
tracing_distance=200.
emd=2601 #13356 #38407

path_output='./out_tracing/'
name_traced=f'EMD{emd}_tr{tracing_distance}nm.em'
motl_trace_input=path_output+name_traced



In [20]:
motl_trace=cryomotl.EmMotl(input_motl=motl_trace_input)
# assume df is your motl DataFrame, with columns ['x','y','z', ...]
N = len(motl_trace.df)


# define your magnitudes; here integers from 1 to 15 (exclusive upper bound if you prefer)
magnitudes = np.arange(1, 30)

# build a dict of noise arrays, one entry per magnitude
noise_dict = {m: make_noise(N, m) for m in magnitudes}

# Example: apply magnitude=5 noise to your df and save new motl
m = 25
df_noisy = motl_trace.df.copy()
df_noisy[['x','y','z']] += noise_dict[m]

motl_tomo_cluster=cryomotl.Motl(motl_df=df_noisy)
motl_tomo_cluster.write_out(f'inputs_noisy/EMD{emd}_noise_{m}.em')
# then write out via cryocat, e.g.:
# from cryocat import motl
# motl.write(df_noisy, 'motl_with_noise_m5.star')

# Angles - NOT WORKING

In [26]:
import numpy as np
import pandas as pd
from scipy.spatial.transform import Rotation

def random_axis_angle_noise(N, angle_deg):
    """
    Return a scipy Rotation containing N random rotations,
    each by exactly 'angle_deg' degrees around a random axis.
    """
    # sample 3D normals and normalize => random unit axes
    axes = np.random.normal(size=(N, 3))
    axes /= np.linalg.norm(axes, axis=1, keepdims=True)
    # build axis-angle array for scipy: each row [ux,uy,uz]*angle_rad
    angle_rad = np.deg2rad(angle_deg)
    axis_angle = axes * angle_rad[:, None] if np.ndim(angle_deg) else axes * angle_rad
    return Rotation.from_rotvec(axis_angle)



motl_trace=cryomotl.EmMotl(input_motl=motl_trace_input)
# assume df is your motl DataFrame, with columns ['x','y','z', ...]
N = len(motl_trace.df)

# assume df has 'phi','psi','theta' in degrees, zxz convention:
angles = motl_trace.df[['phi','psi','theta']].to_numpy()  # shape (N,3)
orig_rots = Rotation.from_euler('zxz', angles, degrees=True)


# create noise rotations for magnitudes 1°…15°
magnitudes = np.arange(1, 16)
noise_rots = {m: random_axis_angle_noise(N, m) for m in magnitudes}

# Example: apply a 7° noise to all particles
m = 1
rot_noisy = noise_rots[m] * orig_rots   # compose: noise ∘ original

# convert back to Euler zxz in degrees, which by default gives
# angles in the ranges you specified:
new_angles = rot_noisy.as_euler('zxz', degrees=True)

# insert back into dataframe (and wrap to the proper ranges, if needed)
df_noisy = motl_trace.df.copy()
df_noisy[['phi','psi','theta']] = new_angles
# phi,theta come out in [-180,180], psi in [0,180] by scipy’s convention

# Finally, save with cryocat:
motl_tomo_cluster=cryomotl.Motl(motl_df=df_noisy)
motl_tomo_cluster.write_out(f'inputs_noisy/EMD{emd}_noise_angle_{m}deg.em')